# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RawanMohamed16/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

Lane: **Refresh / content opportunity scoring** — rank content items for human review when search demand looks like it is softening.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `writing-data-contracts` + `flyrank/flyrank-data`.

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn

In [ ]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata

        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Hugging Face READ token (hf_...): ")

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

## 1. Unit of analysis + time window

**The contract (plain words):**

1. **One row means:** one pseudonymized **content item** (`client_hash_id` + `content_hash_id`) observed on **one calendar day** in the daily fact table; for modeling I collapse March days to **one row per content item** with features from the first half of the month.
2. **Table(s):** `fact_content_daily_performance` (March 2026 partition), with optional joins to `dim_content` / `dim_clients` for metadata and coverage checks only.
3. **Time window:** **March 2026** (`month=2026-03`). **Features** use **2026-03-01 → 2026-03-15** (knowable by end of day 15). **Label proxy** compares **2026-03-16 → 2026-03-31** impressions to the first half (decision-support decline flag, not causal proof).
4. **Predict / rank:** whether **GSC impressions fall materially** in the second half of March vs the first half (`is_declining_mar` — rule-based proxy for “needs a refresh look”).
5. **Deliberately excluding:** rows where **`ga4_data_available IS NOT TRUE`** when I use GA4 fields (zeros there mean “not tracked yet,” not zero traffic); **hash IDs** as model inputs; and **query-table `*_last30` fields** when the label window overlaps the query snapshot (leakage risk per data dictionary).

In [ ]:
# Quick window check on the March partition
con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items
    FROM {FACT_MAR}
""").show()

## 2. Fields: feature / label / context / excluded

| Bucket | Fields |
|---|---|
| **Label / proxy** | `is_declining_mar` (derived from 2nd-half vs 1st-half `gsc_impressions` in March) |
| **Features (max 5 used below)** | `log_imp_first_half`, `log_clk_first_half`, `avg_pos_first_half`, `ctr_first_half_pct`, `active_gsc_days_first_half` |
| **Context (join/split only)** | `client_hash_id`, `content_hash_id`, `report_date`, `dim_content.content_type` |
| **Excluded** | `client_hash_id` / `content_hash_id` as model inputs (pseudonyms for grouping only); GA4 metrics when `ga4_data_available IS NOT TRUE`; any label-period impression totals as features (leakage); product-style composite scores (not in release) |

In [ ]:
# Missingness pattern on GA4 flag in March (why we use IS TRUE / IS NOT TRUE)
con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS rows,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM {FACT_MAR}
    GROUP BY 1
    ORDER BY rows DESC
""").show()

## 3. Verify it with queries (grain, counts, missing values, windows)

Three proof queries on **`month=2026-03`**, then a small feature frame, then the leakage trap.

### Query A — Grain (daily fact)

At daily grain, `(report_date, client_hash_id, content_hash_id)` should be unique.

In [ ]:
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").show()

### Query B — Scale

Row count and date span for the March slice.

In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) AS daily_rows,
        MIN(report_date) AS first_day,
        MAX(report_date) AS last_day,
        DATE_DIFF('day', MIN(report_date), MAX(report_date)) + 1 AS calendar_days
    FROM {FACT_MAR}
""").show()

### Query C — Availability (`IS TRUE`)

How many March rows have **usable GA4** vs the rest.

In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_usable_rows,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*),
            2
        ) AS pct_ga4_usable
    FROM {FACT_MAR}
""").show()

### Five features (March, content-level)

Each feature is justified with **knowable at the decision moment because…**

In [ ]:
FEATURE_SQL = f"""
WITH daily AS (
    SELECT *
    FROM {FACT_MAR}
),
item_march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clk_first_half,
        AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_avg_position > 0
                 THEN gsc_avg_position END) AS avg_pos_first_half,
        COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0
                          THEN report_date END) AS active_gsc_days_first_half
    FROM daily
    GROUP BY 1, 2
    HAVING imp_first_half >= 50
)
SELECT
    client_hash_id,
    content_hash_id,
    imp_first_half,
    imp_second_half,
    LN(1 + imp_first_half) AS log_imp_first_half,
    LN(1 + clk_first_half) AS log_clk_first_half,
    avg_pos_first_half,
    CASE WHEN imp_first_half > 0
         THEN 100.0 * clk_first_half / imp_first_half END AS ctr_first_half_pct,
    active_gsc_days_first_half,
    CASE WHEN imp_second_half < 0.8 * imp_first_half THEN 1 ELSE 0 END AS is_declining_mar
FROM item_march
"""

features = con.sql(FEATURE_SQL).df()
print(f"content items in March feature frame: {len(features):,}")
print(f"decline base rate: {features['is_declining_mar'].mean():.3f}")
features.head()

**Feature justifications**

- `log_imp_first_half` — knowable at the decision moment because it sums GSC impressions only through **2026-03-15**.
- `log_clk_first_half` — knowable because clicks through **2026-03-15** are already reported in Search Console lag.
- `avg_pos_first_half` — knowable because average position through **2026-03-15** is historical GSC, not forward-looking.
- `ctr_first_half_pct` — knowable because it is clicks ÷ impressions from the same closed first-half window (rates here are on a 0–100 style scale when interpreted as percent).
- `active_gsc_days_first_half` — knowable because it counts distinct days through **2026-03-15** with observed GSC impressions only (no forward days).

### The trap (leakage lesson)

I **deliberately** add `imp_second_half` — it is built from the same days the label uses. The model should look artificially strong; then I drop it.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

HONEST_COLS = [
    "log_imp_first_half",
    "log_clk_first_half",
    "avg_pos_first_half",
    "ctr_first_half_pct",
    "active_gsc_days_first_half",
]
LEAK_COL = "imp_second_half"

model_df = features.dropna(subset=HONEST_COLS + [LEAK_COL, "is_declining_mar"]).copy()
y = model_df["is_declining_mar"]


def quick_auc(cols):
    X = model_df[cols]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    clf = RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced"
    )
    clf.fit(X_tr, y_tr)
    prob = clf.predict_proba(X_te)[:, 1]
    return roc_auc_score(y_te, prob)


auc_honest = quick_auc(HONEST_COLS)
auc_leak = quick_auc(HONEST_COLS + [LEAK_COL])
print(f"ROC-AUC (honest features only): {auc_honest:.3f}")
print(f"ROC-AUC (with leaked {LEAK_COL}): {auc_leak:.3f}")
print(f"AUC jump from leakage: {auc_leak - auc_honest:+.3f}")

In [ ]:
# Keep the honest number — leaked column removed from training view
honest_frame = model_df[HONEST_COLS + ["is_declining_mar", "client_hash_id", "content_hash_id"]]
print(honest_frame.shape)
honest_frame.head()

## 4. Data limits

**Named limitation — unbalanced panel / uneven client history:** `dim_clients.gsc_data_start` and `ga4_data_start` differ by client, so a single global calendar window (even March 2026) mixes clients with **different amounts of trustworthy history**. A decline flag in March for a client whose tracking started mid-month is not comparable to a client with 12+ months of GSC — the contract filters GA4 with `IS TRUE`, but it **cannot** fully fix unequal depth without per-client windows or grouped validation.

In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) AS clients,
        MIN(gsc_data_start) AS earliest_gsc,
        MAX(gsc_data_start) AS latest_gsc,
        COUNT(*) FILTER (WHERE gsc_data_start > DATE '2026-01-01') AS clients_gsc_starts_after_2026
    FROM {DIM_CLIENTS}
""").show()

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run in Colab after setting `HF_TOKEN` secret**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.